In [1]:
from sage.all import *

class CantorZassenhaus:
    """
    Sonlu cisimler üzerinde Cantor-Zassenhaus Çarpanlara Ayırma
    Algoritmasının adım adım izlenebilir implementasyonu.

    Cantor-Zassenhaus algoritması, GF(q) üzerinde tanımlı bir f(x)
    polinomunu (q tek asal kuvveti olmak üzere) indirgenemez
    çarpanlarına ayırmak için iki aşamalı bir yapı kullanır:

      (1) DDF (Distinct-Degree Factorization / Farklı Dereceli
          Ayrıştırma): f(x)'i, aynı dereceye sahip indirgenemez
          çarpanların çarpımından oluşan gruplara ayırır. Bu aşama,
          x^(q^d) ≡ x (mod h(x)) özelliğini sağlayan derece-d
          indirgenemez polinomların cebirsel karakterizasyonuna dayanır.

      (2) EDF (Equal-Degree Factorization / Eşit Dereceli Ayrıştırma):
          DDF aşamasında elde edilen, aynı dereceden birden fazla
          indirgenemez çarpan içeren grupları, rastgele polinom
          seçimine dayalı olasılıksal bir yöntemle (Cantor-Zassenhaus
          rastgele bölme adımı) tek tek çarpanlarına ayırır.

    Bu implementasyon yalnızca q'nun tek olduğu (karakteristik p != 2)
    klasik Cantor-Zassenhaus varyantını uygular; q çift durumunda
    farklı bir EDF stratejisi (trace tabanlı yöntemler) gerekmektedir.

    Referans: Tez, Bölüm 4 — Tek Değişkenli Polinomların Çarpanlara
    Ayrılması (DDF ve EDF Algoritmaları)
    """

    def __init__(self, field):
        """
        Sınıfı verilen sonlu cisim üzerinde başlatır.

        Parametreler
        ------------
        field : SageMath sonlu cismi (GF(q))
            Polinomların katsayılarının ait olduğu sonlu cisim.
            Bu implementasyonun geçerliliği için q'nun tek olması
            (yani cismin karakteristiğinin 2'den farklı bir asal
            olması) gerekmektedir; bu koşul check_input() içinde
            ayrıca denetlenir.
        """
        self.F = field
        self.q = field.order()                  # Cismin eleman sayısı q = p^m
        self.R = PolynomialRing(self.F, 'x')     # F_q[x] tek değişkenli polinom halkası
        self.x = self.R.gen()                    # Halkanın üreteci (belirsiz) x

        print("\n" + "="*70)
        print("CANTOR-ZASSENHAUS ALGORİTMASI")
        print(f"Cisim: GF({self.q})")
        print("="*70)

    # ----------------------------------------------------------------
    # ADIM 1: GİRDİ KONTROLÜ
    # ----------------------------------------------------------------
    def check_input(self, f):
        """
        Girdi polinomunun Cantor-Zassenhaus algoritmasının ön
        koşullarını sağlayıp sağlamadığını denetler ve polinomu
        monik forma dönüştürür.

        Kontrol edilen ön koşullar:
          (i)   f sıfır polinom olmamalıdır.
          (ii)  f sabit (derece <= 0) olmamalıdır.
          (iii) Cisim karakteristiği tek olmalıdır (q mod 2 != 0);
                bu implementasyon yalnızca q tek iken geçerlidir,
                çünkü EDF adımında kullanılan (q^d - 1)/2 üssü ancak
                q tek olduğunda tam sayı üretir.
          (iv)  f kare çarpansız (square-free) olmalıdır; aksi hâlde
                EBOB tabanlı DDF/EDF adımları katlı kökler nedeniyle
                yanlış sonuç üretebilir.

        Parametreler
        ------------
        f : F_q[x] polinomu
            Kontrol edilecek ve gerekirse monikleştirilecek polinom.

        Döndürür
        --------
        F_q[x] polinomu veya None
            Tüm ön koşullar sağlanıyorsa monik hale getirilmiş f;
            herhangi bir ön koşul sağlanmıyorsa None.
        """
        print("\n[ADIM 1] Girdi Kontrolü Başlıyor...")

        # Sıfır polinom, çarpanlara ayırma açısından tanımsızdır
        if f == 0:
            print("   -> HATA: Sıfır polinom için çarpanlara ayırma yapılmaz.")
            return None

        # Sabit (derece 0) polinomların indirgenemez çarpanı yoktur
        if f.degree() <= 0:
            print("   -> HATA: Sabit polinom için Cantor-Zassenhaus uygulanmaz.")
            return None

        # Bu sürüm klasik q tek Cantor-Zassenhaus EDF adımını kullanır.
        # q çift olduğunda (yani karakteristik p=2 olduğunda) EDF adımında
        # kullanılan (q^d - 1)/2 üssü tam sayı olmaz; bu durumda iz
        # (trace) tabanlı alternatif bir EDF yöntemi gereklidir.
        if self.q % 2 == 0:
            print("   -> HATA: Bu implementasyon q tek iken çalışan klasik Cantor-Zassenhaus sürümüdür.")
            print("   -> q çift ise farklı bir EDF yaklaşımı gerekir.")
            return None

        # Monik hale getir: algoritmanın sonraki adımlarında (özellikle
        # EBOB hesaplarında) baş katsayının 1 olması işlemleri sadeleştirir
        lc = f.leading_coefficient()
        f_monic = f / lc

        print(f"   -> Girdi polinomu: {f}")
        print(f"   -> Baş katsayı: {lc}")
        print(f"   -> Monik hale getirilmiş polinom: {f_monic}")

        # Square-free kontrolü: türev hesabı
        # Karakteristik p'de x^(kp) biçimindeki terimlerin türevi sıfır
        # olduğundan, f'(x) = 0 olması f(x)'in p'nin katı üslerden
        # oluştuğuna (yani f = g^p biçiminde olduğuna) işaret eder.
        df = f_monic.derivative()
        print(f"   -> Türev f'(x): {df}")

        if df == 0:
            print("   -> Türev 0! Polinom p. kuvvet formundadır.")
            print("   -> Bu polinom square-free değildir.")
            print("   -> Cantor-Zassenhaus algoritmasından önce Square-Free Factorization uygulanmalıdır.")
            return None

        # EBOB(f, f') = 1 koşulu, f'nin katlı köke sahip olmadığının
        # cebirsel testidir; f ile f' ortak bir kökü paylaşıyorsa bu
        # kök f'de katlıdır.
        d = gcd(f_monic, df)
        print(f"   -> EBOB(f, f') = {d}")

        if d != 1:
            print("   -> EBOB(f, f') != 1. Polinom square-free değildir.")
            print("   -> Cantor-Zassenhaus algoritması square-free polinomlar üzerinde çalışır.")
            print("   -> Önce Square-Free Factorization algoritmasını uygulayın.")
            return None

        print("   -> Polinom monik ve square-free.")
        print("   -> Cantor-Zassenhaus algoritması bu polinom üzerinde çalıştırılacaktır.")

        return f_monic

    # ----------------------------------------------------------------
    # ADIM 2: DDF
    # ----------------------------------------------------------------
    def distinct_degree_factorization(self, f):
        """
        Farklı Dereceli Ayrıştırma (Distinct-Degree Factorization, DDF)
        adımını uygular.

        Teorik dayanak: F_q[x] halkasında, x^(q^d) - x polinomu,
        derecesi d'yi bölen tüm indirgenemez polinomların çarpımına
        eşittir. Bu nedenle EBOB(x^(q^d) - x, f_curr) hesaplanarak,
        f_curr'nin içindeki derecesi tam olarak d olan (veya d'yi
        bölen) indirgenemez çarpanların ürünü elde edilebilir.

        Algoritma, d = 1, 2, 3, ... şeklinde artan derecelerde tarama
        yaparak her adımda:
          - h(x) = x^(q^d) mod f_curr değerini günceller
            (tekrarlı kare alma ile modüler üs hesaplama),
          - g = EBOB(h(x) - x, f_curr) hesaplar,
          - g != 1 ise, derece-d indirgenemez çarpanların çarpımı olan
            g'yi sonuç listesine ekler ve f_curr'yi g'ye bölerek küçültür.

        Döngü, 2d, kalan polinomun derecesini aştığında sona erer;
        çünkü bu noktada kalan polinom artık tek bir derece grubuna
        (kendi derecesine eşit) karşılık gelmek zorundadır.

        Parametreler
        ------------
        f : F_q[x] polinomu
            check_input() tarafından doğrulanmış monik, kare çarpansız polinom.

        Döndürür
        --------
        list of (int, F_q[x])
            Her biri (d, R_d(x)) biçiminde bir liste; burada d bir
            derece değeri, R_d(x) ise f'nin derecesi d olan tüm
            indirgenemez çarpanlarının çarpımıdır.
        """
        print(f"\n[ADIM 2] DDF: Farklı Dereceli Ayrıştırma Başlıyor...")
        print(f"   Polinom: {f}")

        result = []
        h = self.x        # h, x^(q^d) mod f_curr değerini tutan yardımcı değişken
        f_curr = f         # f_curr, henüz gruplanmamış kalan çarpanların çarpımı
        d = 0

        while f_curr != 1:
            d += 1

            print(f"\n   --- DDF Döngüsü: d = {d} ---")

            # Eğer 2d, kalan polinomun derecesini aşıyorsa, kalan
            # polinomun kendisi tek bir derece grubuna karşılık gelir
            # (çünkü daha küçük dereceli tüm çarpanlar zaten önceki
            # adımlarda ayıklanmıştır ve kalan derece 2d'den küçüktür,
            # dolayısıyla en fazla bir indirgenemez çarpan bulunabilir
            # ya da kalanın tamamı tek bir dereceye aittir).
            if 2*d > f_curr.degree():
                print("   -> Kalan polinom artık tek bir derece grubuna karşılık gelir.")
                print(f"   -> Kalan grup derecesi: {f_curr.degree()}")
                print(f"   -> Kalan grup polinomu: {f_curr}")
                result.append((f_curr.degree(), f_curr))
                break

            # h = x^(q^d) mod f_curr olacak şekilde güncellenir.
            # pow(h, q, f_curr) çağrısı, önceki adımdaki h = x^(q^(d-1))
            # değerini q. kuvvete yükseltip f_curr'e göre indirger;
            # böylece her adımda x^(q^d)'yi baştan hesaplamak yerine
            # bir önceki sonuçtan hızlıca türetilir.
            h = pow(h, self.q, f_curr)
            print(f"   -> h = x^(q^{d}) mod f_curr = {h}")

            # g = EBOB(h(x) - x, f_curr): derece-d indirgenemez
            # çarpanların (varsa) çarpımını verir
            g = gcd(h - self.x, f_curr)
            print(f"   -> g = EBOB(h - x, f_curr) = {g}")

            if g != 1:
                print(f"   -> Derecesi {d} olan indirgenemez çarpanların çarpımı bulundu:")
                print(f"      R_{d}(x) = {g}")

                result.append((d, g))

                # Bulunan grup f_curr'den çıkarılır; kalan işlem
                # yalnızca henüz gruplanmamış çarpanlar üzerinde sürer
                f_curr = f_curr // g
                print(f"   -> Kalan polinom f_curr = {f_curr}")

                # f_curr küçüldüğü için h de yeni f_curr'e göre
                # yeniden indirgenmelidir; aksi hâlde bir sonraki
                # adımdaki modüler üs hesabı yanlış modülde yapılır
                if f_curr != 1:
                    h = h % f_curr

        print("\n   DDF Sonucu:")
        for d, poly in result:
            print(f"      Derece grubu d={d}: {poly}")

        return result

    # ----------------------------------------------------------------
    # ADIM 3: EDF / CANTOR-ZASSENHAUS
    # ----------------------------------------------------------------
    def equal_degree_factorization(self, f, d):
        """
        Eşit Dereceli Ayrıştırma (Equal-Degree Factorization, EDF)
        adımını, klasik Cantor-Zassenhaus rastgele bölme yöntemiyle
        uygular.

        Teorik dayanak: f(x), her biri derece d olan k adet farklı
        indirgenemez çarpanın çarpımı olsun (f = h1 * h2 * ... * hk).
        Rastgele seçilen bir a(x) polinomu için Çin Kalan Teoremi
        (CRT) sayesinde, F_q[x]/(f) bölüm halkası
        F_q[x]/(h1) x F_q[x]/(h2) x ... x F_q[x]/(hk)
        halkasına izomorftur. q tek olduğundan, her bir bileşende
        a(x)^((q^d - 1)/2) değeri bağımsız olarak +1, -1 veya 0
        değerlerinden birini alır (Öklid/ikinci dereceden kalan
        argümanının polinom halkalarına genellemesi). Bu değerlerin
        bileşenler arasında farklılık göstermesi olasılığı yüksek
        olduğundan, b(x) = a(x)^((q^d-1)/2) - 1 polinomunun f ile
        EBOB'u, çarpanların bir kısmını (aşikâr olmayan bir bölen
        olarak) ayırır.

        Algoritma, hedeflenen çarpan sayısına ulaşılana veya maksimum
        deneme sayısı aşılana kadar rastgele a(x) seçimini tekrarlar.

        Parametreler
        ------------
        f : F_q[x] polinomu
            DDF aşamasından gelen, tüm indirgenemez çarpanları aynı
            dereceden (d) olan monik polinom grubu.
        d : int
            Bu gruptaki indirgenemez çarpanların ortak derecesi.

        Döndürür
        --------
        list of F_q[x]
            f'nin derece-d indirgenemez çarpanlarının listesi
            (her biri monik hale getirilmiş).
        """
        print(f"\n[ADIM 3] EDF: Eşit Dereceli Ayrıştırma Başlıyor...")
        print(f"   Polinom: {f}")
        print(f"   Hedef indirgenemez çarpan derecesi: d = {d}")

        # Grup polinomu monik hale getirilir
        f = f / f.leading_coefficient()

        # Bu gruptan beklenen indirgenemez çarpan sayısı:
        # toplam derece / her bir çarpanın derecesi
        num_factors = f.degree() // d
        print(f"   Beklenen çarpan sayısı: {num_factors}")

        # Grup zaten tek bir indirgenemez çarpandan oluşuyorsa,
        # rastgele bölme adımına hiç gerek yoktur
        if num_factors == 1:
            print("   -> Bu grup zaten tek bir indirgenemez çarpandan oluşuyor.")
            return [f]

        # Cantor-Zassenhaus'un temel üssü: (q^d - 1)/2.
        # q tek olduğu için q^d - 1 çifttir ve bu ifade her zaman
        # tam sayı üretir; bu üs, F_(q^d) cisminin çarpımsal grubunun
        # yarısına (ikinci dereceden kalanlar alt grubuna) karşılık gelir.
        exponent = (self.q**d - 1) // 2
        print(f"   Kullanılan üs: (q^d - 1)/2 = ({self.q}^{d} - 1)/2 = {exponent}")

        factors = [f]
        trial = 0
        max_trials = 200   # Olasılıksal algoritmanın sonsuz döngüye girmesini önleyen güvenlik sınırı

        while len(factors) < num_factors and trial < max_trials:
            trial += 1
            print(f"\n   --- EDF Rastgele Deneme {trial} ---")

            # Rastgele bir a(x) polinomu seç; derece f.degree()-1'den
            # küçük tutulur ki F_q[x]/(f) bölüm halkasının bir elemanını
            # temsil etsin (f'e göre zaten indirgenmiş kabul edilir)
            a = self.R.random_element(degree=f.degree() - 1)

            if a == 0:
                print("      Rastgele polinom 0 geldi. Tekrar denenecek.")
                continue

            print(f"      Rastgele seçilen a(x): {a}")

            new_factors = []

            for u in factors:
                # Bu parça zaten hedef dereceye ulaşmışsa (yani tek bir
                # indirgenemez çarpansa), daha fazla bölünmeye gerek yoktur
                if u.degree() == d:
                    print(f"      -> {u} zaten hedef derece {d} değerinde. Korunuyor.")
                    new_factors.append(u)
                    continue

                print(f"\n      İncelenen parça u(x): {u}")

                # Önce şanslı durum kontrolü: gcd(a,u).
                # Eğer a(x), u'nun bir çarpanı ile tesadüfen ortak kök
                # paylaşıyorsa, Cantor-Zassenhaus'un asıl adımına
                # (üs alma işlemine) gerek kalmadan doğrudan bir bölen
                # elde edilebilir.
                g0 = gcd(a, u)
                print(f"      g0 = EBOB(a,u) = {g0}")

                if g0 != 1 and g0 != u:
                    print("      [ŞANSLI AYRIŞMA] a(x), u(x) ile aşikar olmayan ortak bölen verdi.")
                    print(f"         g0    = {g0}")
                    print(f"         u/g0  = {u // g0}")

                    new_factors.append(g0.monic())
                    new_factors.append((u // g0).monic())
                    continue

                # Cantor-Zassenhaus ana adımı:
                # b(x) = a(x)^((q^d-1)/2) mod u, sonra 1 çıkarılır.
                # u'nun her bir indirgenemez alt-çarpanında b(x) değeri
                # bağımsız olarak +1, -1 veya 0'a eşit olabileceğinden,
                # b(x) - 1 ile u'nun EBOB'u, u'yu aşikâr olmayan bir
                # şekilde bölme olasılığı yüksek bir adaydır.
                b = pow(a, exponent, u) - 1
                g = gcd(b, u)

                print(f"      b(x) = a(x)^(({self.q}^{d}-1)/2) - 1 mod u")
                print(f"      g = EBOB(b,u) = {g}")

                if g != 1 and g != u:
                    print("      [AYRIŞTI] Aşikar olmayan çarpan bulundu.")
                    print(f"         g    = {g}")
                    print(f"         u/g  = {u // g}")

                    new_factors.append(g.monic())
                    new_factors.append((u // g).monic())
                else:
                    # Bu denemede seçilen a(x), u'yu ayrıştırmaya
                    # yetmedi; u değişmeden bir sonraki denemeye kalır
                    print("      Bu denemede bu parça ayrışmadı.")
                    new_factors.append(u)

            factors = new_factors
            print(f"\n   Güncel çarpan sayısı: {len(factors)} / {num_factors}")

        if len(factors) < num_factors:
            print("\n   [UYARI] Maksimum deneme sayısına ulaşıldı.")
            print("   Daha fazla deneme için max_trials değeri artırılabilir.")

        print("\n   EDF Sonucu:")
        for fac in factors:
            print(f"      {fac}")

        return [fac.monic() for fac in factors]

    # ----------------------------------------------------------------
    # ANA AKIŞ
    # ----------------------------------------------------------------
    def run(self, f):
        """
        Girdi polinomu üzerinde uçtan uca Cantor-Zassenhaus iş akışını
        yürütür.

        İş akışı:
          1. check_input() ile ön koşullar (sıfır/sabit polinom
             kontrolü, q'nun tek olması, kare çarpansızlık) doğrulanır.
          2. distinct_degree_factorization() ile f, aynı dereceden
             indirgenemez çarpanların gruplarına ayrılır (DDF).
          3. Her DDF grubu için, grup zaten tek bir indirgenemez
             çarpandan oluşuyorsa doğrudan kabul edilir; birden fazla
             çarpan içeriyorsa equal_degree_factorization() ile
             (EDF) tek tek ayrıştırılır.
          4. Elde edilen tüm indirgenemez çarpanlar dereceye göre
             sıralanır ve çarpımları hesaplanarak orijinal f ile
             karşılaştırılır (doğruluk sağlaması).

        Parametreler
        ------------
        f : F_q[x] polinomu
            Çarpanlarına ayrılacak polinom.

        Döndürür
        --------
        list of F_q[x]
            f'nin tüm indirgenemez çarpanlarının (dereceye göre
            sıralanmış) listesi; ön koşullar sağlanmıyorsa boş liste.
        """
        print(f"\n>>> İŞLEM BAŞLIYOR")
        print(f"Girdi polinomu: {f}")

        # 1. Girdi kontrolü ve monikleştirme
        f = self.check_input(f)

        if f is None:
            print("\n" + "="*70)
            print("İŞLEM DURDURULDU")
            print("="*70)
            return []

        final_factors = []

        # 2. DDF: f'i aynı dereceden çarpan gruplarına ayır
        ddf_groups = self.distinct_degree_factorization(f)

        # 3. Her DDF grubu için EDF uygula
        for d, group_poly in ddf_groups:
            print("\n" + "-"*70)
            print(f"DDF grubundan EDF adımına geçiliyor: d = {d}")
            print(f"Grup polinomu: {group_poly}")
            print("-"*70)

            if group_poly.degree() == d:
                # Grup derecesi, hedef dereceye eşitse zaten tek bir
                # indirgenemez çarpandır; EDF'e gerek yoktur
                print("   -> Grup zaten tek bir indirgenemez çarpandan oluşuyor.")
                final_factors.append(group_poly.monic())
            else:
                # Grup birden fazla aynı dereceden çarpan içeriyor;
                # rastgele bölme yöntemiyle (EDF) ayrıştırılmalıdır
                primes = self.equal_degree_factorization(group_poly, d)
                final_factors.extend(primes)

        # Sonuçlar okunabilirlik amacıyla artan dereceye göre sıralanır
        final_factors.sort(key=lambda p: p.degree())

        print("\n" + "="*70)
        print("CANTOR-ZASSENHAUS SONUCU")
        print("="*70)

        check_poly = self.R(1)

        for i, p in enumerate(final_factors, start=1):
            print(f"   {i}. Çarpan: {p}  [Derece: {p.degree()}]")
            check_poly *= p

        print("-" * 70)
        print(f"Çarpanların çarpımı: {check_poly}")
        print(f"Girdi monik polinom : {f}")

        # Doğruluk sağlaması: bulunan çarpanların çarpımı, monikleştirilmiş
        # girdi polinomuna tam olarak eşit olmalıdır
        if check_poly == f:
            print("SAĞLAMA: BAŞARILI")
        else:
            print("SAĞLAMA: HATA!")

        print("="*70)

        return final_factors




In [2]:
# ============================================================
# SENARYO 1: Manuel cisim ve manuel polinom
# ============================================================
# GF(5) üzerinde, farklı derecelerden (1, 1, 1, 2, 3) bilinen
# indirgenemez çarpanların çarpımından elle bir polinom oluşturulur.
# Bu senaryo, hem DDF'nin aynı dereceden birden fazla çarpanı (üç
# adet derece-1 çarpan) doğru şekilde gruplandırdığını hem de EDF'nin
# bu grubu doğru şekilde ayrıştırdığını gözlemlemeye olanak tanır.

q = 5
F = GF(q)

solver = CantorZassenhaus(F)
R = solver.R
x = solver.x

# Manuel polinom girişi
# Bu örnek GF(5) üzerinde square-free olacak şekilde seçilmiştir.
f = (x+1)*(x+2)*(x+3)*(x**2 + x + 2) * (x**3 + 2*x + 1)


print("\nManuel girilen polinom:")
print(f"Açık hali: {f}")
print(f"deg(f) = {f.degree()}")

solver.run(f)




CANTOR-ZASSENHAUS ALGORİTMASI
Cisim: GF(5)

Manuel girilen polinom:
Açık hali: x^8 + 2*x^7 + x^6 + 4*x^5 + 3*x^4 + 4*x^3 + 2*x + 2
deg(f) = 8

>>> İŞLEM BAŞLIYOR
Girdi polinomu: x^8 + 2*x^7 + x^6 + 4*x^5 + 3*x^4 + 4*x^3 + 2*x + 2

[ADIM 1] Girdi Kontrolü Başlıyor...
   -> Girdi polinomu: x^8 + 2*x^7 + x^6 + 4*x^5 + 3*x^4 + 4*x^3 + 2*x + 2
   -> Baş katsayı: 1
   -> Monik hale getirilmiş polinom: x^8 + 2*x^7 + x^6 + 4*x^5 + 3*x^4 + 4*x^3 + 2*x + 2
   -> Türev f'(x): 3*x^7 + 4*x^6 + x^5 + 2*x^3 + 2*x^2 + 2
   -> EBOB(f, f') = 1
   -> Polinom monik ve square-free.
   -> Cantor-Zassenhaus algoritması bu polinom üzerinde çalıştırılacaktır.

[ADIM 2] DDF: Farklı Dereceli Ayrıştırma Başlıyor...
   Polinom: x^8 + 2*x^7 + x^6 + 4*x^5 + 3*x^4 + 4*x^3 + 2*x + 2

   --- DDF Döngüsü: d = 1 ---
   -> h = x^(q^1) mod f_curr = x^5
   -> g = EBOB(h - x, f_curr) = x^3 + x^2 + x + 1
   -> Derecesi 1 olan indirgenemez çarpanların çarpımı bulundu:
      R_1(x) = x^3 + x^2 + x + 1
   -> Kalan polinom f_cur

[x + 3, x + 1, x + 2, x^2 + x + 2, x^3 + 2*x + 1]

In [3]:
# ============================================================
# SENARYO 2: Derece listesine göre rastgele çarpanlı polinom
# ============================================================
# GF(5) üzerinde, 'degrees' listesinde belirtilen derecelerde
# rastgele seçilmiş indirgenemez çarpanların çarpımından bir polinom
# üretilir. İki adet derece-3 çarpanın varlığı, DDF'nin bu grubu
# tek bir R_3(x) polinomunda topladığını ve ardından EDF'nin bu
# grubu iki ayrı çarpana böldüğünü test etmeye olanak tanır.

q = 5
F = GF(q)

solver = CantorZassenhaus(F)
R = solver.R
x = solver.x

# Her sayı üretilecek indirgenemez çarpanın derecesidir.Liste uzatılabilir
degrees = [3, 3, 2, 1]

f = R(1)
factors = []

for d in degrees:
    g = R.irreducible_element(d, algorithm="random")

    # Aynı indirgenemez çarpanın birden fazla seçilmesi f'yi katlı
    # köklü hale getirir (square-free koşulunu bozar); bu nedenle
    # zaten seçilmiş bir çarpan tekrar çıkarsa yeniden örneklenir.
    while g in factors:
        g = R.irreducible_element(d, algorithm="random")

    factors.append(g)
    f *= g

print("\nSeçilen indirgenemez çarpanlar:")
for g in factors:
    print(f"   {g}")

print("\nOluşturulan polinom:")
print(f"f(x) = {f}")
print(f"deg(f) = {f.degree()}")

solver.run(f)


CANTOR-ZASSENHAUS ALGORİTMASI
Cisim: GF(5)

Seçilen indirgenemez çarpanlar:
   x^3 + 2*x^2 + x + 3
   x^3 + 2*x^2 + x + 4
   x^2 + 3*x + 4
   x + 3

Oluşturulan polinom:
f(x) = x^9 + 3*x^7 + x^6 + 2*x^5 + 2*x^4 + x^3 + 3*x^2 + 4
deg(f) = 9

>>> İŞLEM BAŞLIYOR
Girdi polinomu: x^9 + 3*x^7 + x^6 + 2*x^5 + 2*x^4 + x^3 + 3*x^2 + 4

[ADIM 1] Girdi Kontrolü Başlıyor...
   -> Girdi polinomu: x^9 + 3*x^7 + x^6 + 2*x^5 + 2*x^4 + x^3 + 3*x^2 + 4
   -> Baş katsayı: 1
   -> Monik hale getirilmiş polinom: x^9 + 3*x^7 + x^6 + 2*x^5 + 2*x^4 + x^3 + 3*x^2 + 4
   -> Türev f'(x): 4*x^8 + x^6 + x^5 + 3*x^3 + 3*x^2 + x
   -> EBOB(f, f') = 1
   -> Polinom monik ve square-free.
   -> Cantor-Zassenhaus algoritması bu polinom üzerinde çalıştırılacaktır.

[ADIM 2] DDF: Farklı Dereceli Ayrıştırma Başlıyor...
   Polinom: x^9 + 3*x^7 + x^6 + 2*x^5 + 2*x^4 + x^3 + 3*x^2 + 4

   --- DDF Döngüsü: d = 1 ---
   -> h = x^(q^1) mod f_curr = x^5
   -> g = EBOB(h - x, f_curr) = x + 3
   -> Derecesi 1 olan indirgenemez çar

[x + 3, x^2 + 3*x + 4, x^3 + 2*x^2 + x + 3, x^3 + 2*x^2 + x + 4]